# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a mass spectrometry protein dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset [FAIR²: Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells](https://sen.science/doi/10.71728/senscience.vq4a-28xa)


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# The metadata object is of class mlcroissant.Metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

# Optionally, show published date and other details
print(f"Version: {getattr(metadata, 'version', 'N/A')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their IDs
print("Available Record Sets:")
for rs in metadata.record_sets:
    print(f"  - Name: {rs.name}, @id: {rs.id}")

# For each record set, list its fields and field IDs
print("\nFields per Record Set:")
for rs in metadata.record_sets:
    print(f"Record Set: {rs.name} (@id: {rs.id})")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"    - Field: {field.name}, @id: {field.id}, Type: {getattr(field, 'data_type', 'N/A')}")
    else:
        print("    <No fields found>")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Define the list of record set @id values you want to extract
# Example: from the overview above, pick the primary protein data table. We'll use the first record set as an example.
record_sets = [rs.id for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    print(f"Loading records for record set '@id': {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from record set {record_set_id}")
    else:
        print(f"No records found in {record_set_id}")

if record_sets:
    primary_record_set_id = record_sets[0]
    print(f"Columns for record set '{primary_record_set_id}':")
    print(dataframes[primary_record_set_id].columns.tolist())
    dataframes[primary_record_set_id].head()
else:
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section demonstrates operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare for further analysis.

In [ ]:
# Example EDA: filtering and normalizing a numeric field

# Choose a numeric field and a group field by @id or column name in the dataframe
primary_record_set_id = record_sets[0]
df = dataframes[primary_record_set_id]
fields = {field.id: field for field in metadata.record_sets[0].fields}

# Try to select a numeric field, e.g., 'cr:coverage_percent' or 'cr:mw'
numeric_field_id = None
for field_id, field in fields.items():
    dtype = getattr(field, 'data_type', None)
    # Look for typical float/integer types
    if dtype and ("Float" in dtype or "Integer" in dtype or "Number" in dtype or "Percent" in field_id.lower()):
        if field_id in df.columns:
            numeric_field_id = field_id
            break
# If still not found, fallback to first numeric column
if not numeric_field_id:
    for col in df.columns:
        # Try guessing by column type
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
print(f"Using numeric field '@id': {numeric_field_id}")

# Set a threshold for filtering
threshold = df[numeric_field_id].quantile(0.75) if numeric_field_id else 0
filtered_df = df[df[numeric_field_id] > threshold] if numeric_field_id else df
print(f"Filtered Data: {filtered_df.shape[0]} records where {numeric_field_id} > {threshold}")

# Normalize numeric field
if numeric_field_id:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping by another field (e.g. cr:sample or cr:protein_class)
group_field_id = None
for field_id, field in fields.items():
    dtype = getattr(field, 'data_type', '')
    if (dtype and "Text" in dtype or "String" in dtype or "category" in dtype or "sample" in field_id.lower()) and field_id != numeric_field_id:
        if field_id in df.columns:
            group_field_id = field_id
            break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped by '{group_field_id}':\n", grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8, 5))
if numeric_field_id in df.columns:
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

if group_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(10,6))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR² mass spectrometry dataset using `mlcroissant`. We listed available record sets and fields by their `@id`, extracted tabular data, performed basic filtering and normalization on a numeric field, and visualized distributions.

Key insights included identifying major proteins (or samples) with high measured values, alongside summary statistics grouped by a chosen classification field. 

You can further customize EDA or extend the notebook with statistical testing or machine learning workflows using the preprocessed DataFrames.
